# Fusion des interventions interrompues
- fonction de fusion
- test avec fonction plus "souple" -> éviter
- comparaison

In [1]:
import pandas as pd

# Charger le df concaténé des deux législatures
df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

print("Shape du df chargé : ", df.shape)

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["code_parole"] = df["code_parole"].fillna("non_précisé")

# # NOTE : après test ne semble pas dramatique de ne pas prendre en compte
# # le df["id_orateur"] = "PA" + df["id_orateur"] : seules 0 ou 2 lignes changent (si code parole ou pas)
# # les recodages "manuels" de PA repérés par ailleurs (voir autre notebook)
# # ne changent rien non plus ici
# # cf surtout des interruptions et ne change pas grand chose au regroup d'interventions
# # + quand erreur pas forcément de changement d'ID entre ou d'interv.

# # Mais par principe si on veut garder :
# # Stabiliser le id_orateur pour être au format AN
# df["id_orateur"] = "PA" + df["id_orateur"]
# # Remplacer les valeurs manquantes de id_acteur par id_orateur quand disponible
# df["id_acteur_originel"] = df["id_acteur"]  # garder une trace
# df["id_acteur"] = df["id_acteur"].combine_first(df["id_orateur"])

Shape du df chargé :  (1127838, 29)


In [2]:
# TODO voir si devient pas trop conservateur avec en plus le code parole ????

"""
==========================
Regroupe les lignes de l'extraction CSV pour fusionner les interventions
d'un même orateur interrompues par des INTERRUPTION_1_10.

Sortie : un CSV entrelacé avec :
  - une ligne par groupe d'intervention fusionnée (texte concaténé)
  - les informations sur le nombre de fragments, d'interruptions reçues, etc.
  - les lignes INTERRUPTION conservées telles quelles, intercalées dans l'ordre
"""

# ---------------------------------------------------------------------------
# Paramètres
# ---------------------------------------------------------------------------

# Codes considérés comme interruptions (conservés tels quels dans la sortie)
CODES_INTERRUPTION = {"INTERRUPTION_1_10"}

# Colonnes invariantes dans un groupe (on garde la valeur de la 1ère ligne)
COLS_META = [
    "uid",
    "SeanceRef",
    "SessionRef",
    "dateSeance",
    "dateSeanceJour",
    "numSeanceJour",
    "numSeance",
    "typeAssemblee",
    "legislature",
    "session",
    "nomFichierJo",
    "presidentSeance",
    "point_titre",
    "point_type",
    "valeur_ptsodj",
    "ordinal_prise",
    "ordre_absolu_seance",
    "id_acteur",
    "id_mandat",
    "code_grammaire",
    "code_style",
    "code_parole",
    "id_syceron",
    "roledebat",
    "nom_orateur",
    "qualite_orateur",
    "id_orateur",
    "stime",
]

# ---------------------------------------------------------------------------
# Fonction principale
# ---------------------------------------------------------------------------


def regrouper(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prend un DataFrame trié par (uid, ordre_absolu_seance) et retourne
    un DataFrame entrelacé :
      - lignes d'intervention fusionnées (nb_fragments >= 1)
      - lignes d'interruption conservées telles quelles (nb_fragments = NaN)
    """
    cols_utiles = list(dict.fromkeys(COLS_META + ["texte"]))
    work = df[cols_utiles].copy()

    work["uid_norm"] = work["uid"].fillna("").astype(str)
    work["id_acteur_norm"] = work["id_acteur"].fillna("").astype(str)
    work["code_grammaire_norm"] = work["code_grammaire"].fillna("").astype(str)
    work["code_parole_norm"] = work["code_parole"].fillna("").astype(str)  # TODO : test
    work["texte_norm"] = work["texte"].fillna("").astype(str)

    work = work.sort_values(["uid_norm", "ordre_absolu_seance"]).reset_index(drop=True)

    resultats = []  # liste finale (interventions + interruptions)
    groupe = None  # groupe en cours d'accumulation
    buffer_interruptions = []  # interruptions entre deux fragments du même orateur

    def ligne_sortie_depuis_base(base_row: dict) -> dict:
        r = {col: base_row[col] for col in cols_utiles}
        r["nb_fragments"] = pd.NA
        r["nb_interruptions_recues"] = pd.NA
        r["a_ete_interrompu"] = pd.NA
        # r["codes_gram_fragments"] = pd.NA # ie pour traçabilité si enlève condition
        # r["codes_parole_fragments"] = pd.NA # ie pour traçabilité si enlève condition
        r["id_syceron_fragments"] = pd.NA
        # r["changement_code_grammaire"] = pd.NA # ie pour traçabilité si enlève condition
        # r["changement_code_parole"] = pd.NA # ie pour traçabilité si enlève condition
        return r

    def clore_groupe(g: dict) -> dict:
        """
        Finalise un groupe. Les interruptions du buffer seront émises APRÈS dans le flux.
        """
        row = g["premiere_ligne"].copy()
        row["texte"] = " ".join(
            g["textes"]
        )  # on prend les textes norm pour éviter les NaN
        row["nb_fragments"] = g["nb_fragments"]
        row["nb_interruptions_recues"] = g["nb_interruptions_recues"]
        row["a_ete_interrompu"] = g["nb_interruptions_recues"] > 0
        # row["codes_gram_fragments"] = "|".join(g["codes_grammaire"]) # ie pour traçabilité si enlève condition
        # row["codes_parole_fragments"] = "|".join(g["codes_parole"]) # ie pour traçabilité si enlève condition
        row["id_syceron_fragments"] = "|".join(g["codes_syceron"])
        # row["changement_code_grammaire"] = len(set(g["codes_grammaire"])) > 1 # ie pour traçabilité si enlève condition
        # row["changement_code_parole"] = len(set(g["codes_parole"])) > 1 # ie pour traçabilité si enlève condition
        return row

    records = work.to_dict("records")

    for row in records:
        cg = row["code_grammaire_norm"]
        cp = row["code_parole_norm"]  # TODO : test
        acteur_str = row["id_acteur_norm"]
        uid_str = row["uid_norm"]
        syc = str(row["id_syceron"]) if pd.notna(row["id_syceron"]) else ""

        # --- Cas 1 : interruption ---
        if cg in CODES_INTERRUPTION:
            if groupe is not None:
                # L'interruption est dans le contexte d'un groupe ouvert :
                # on l'ajoute au buffer (elle sera émise si le même orateur reprend)
                buffer_interruptions.append(row)
                groupe["nb_interruptions_recues"] += 1
            else:
                # Interruption hors contexte (cas rare) : on l'émet directement
                resultats.append(ligne_sortie_depuis_base(row))
            continue

        # --- Cas 2 : intervention principale ---
        if (
            groupe is not None
            and buffer_interruptions  # on regroupe que si bien interrompu (et pas parle 2 fois de suite)
            and acteur_str != ""  # cf les nan convertis en ""
            and groupe["id_acteur"] == acteur_str
            and groupe["uid"] == uid_str
            and groupe["codes_grammaire"][-1] == cg
            and groupe["codes_parole"][-1] == cp
        ):
            # Même orateur, même séance, mêmes codes, avec interruption -> on fusionne
            groupe["textes"].append(row["texte_norm"])
            groupe["codes_grammaire"].append(cg)
            groupe["codes_parole"].append(cp)  # TODO : test
            groupe["codes_syceron"].append(syc)
            groupe["nb_fragments"] += 1
        else:
            # Nouvel orateur ou nouvelle séance ou changement de code_grammaire
            if groupe is not None:
                # Clore le groupe précédent
                resultats.append(clore_groupe(groupe))
                # Et les interruptions en buffer suivent le groupe
                for irr in buffer_interruptions:
                    resultats.append(ligne_sortie_depuis_base(irr))
                buffer_interruptions = []

            groupe = {
                "uid": uid_str,
                "id_acteur": acteur_str,
                "premiere_ligne": {col: row[col] for col in cols_utiles},
                "textes": [row["texte_norm"]],
                "codes_grammaire": [cg],
                "codes_parole": [cp],
                "codes_syceron": [syc],
                "nb_fragments": 1,
                "nb_interruptions_recues": 0,
            }
    # Clore le dernier groupe
    if groupe is not None:
        resultats.append(clore_groupe(groupe))
        for irr in buffer_interruptions:
            resultats.append(ligne_sortie_depuis_base(irr))

    return pd.DataFrame(resultats)


In [3]:
df_group_bis_CG_CP = regrouper(df)
print("Shape du df regroupé : ", df_group_bis_CG_CP.shape)

df_group_bis_CG_CP.to_csv(
    "../data/interim/TEST_INTERRUPTIONS_BIS_CG_CP.csv", index=False
)

Shape du df regroupé :  (981871, 33)


# TESTS

In [4]:
# NOTE VIRER ENSUITE : FONCTION SANS CODE PAROLE POUR TESTER.

"""
==========================
Regroupe les lignes de l'extraction CSV pour fusionner les interventions
d'un même orateur interrompues par des INTERRUPTION_1_10.

Sortie : un CSV entrelacé avec :
  - une ligne par groupe d'intervention fusionnée (texte concaténé)
  - les informations sur le nombre de fragments, d'interruptions reçues, etc.
  - les lignes INTERRUPTION conservées telles quelles, intercalées dans l'ordre
"""

# ---------------------------------------------------------------------------
# Paramètres
# ---------------------------------------------------------------------------

# Codes considérés comme interruptions (conservés tels quels dans la sortie)
CODES_INTERRUPTION = {"INTERRUPTION_1_10"}

# Colonnes invariantes dans un groupe (on garde la valeur de la 1ère ligne)
COLS_META = [
    "uid",
    "SeanceRef",
    "SessionRef",
    "dateSeance",
    "dateSeanceJour",
    "numSeanceJour",
    "numSeance",
    "typeAssemblee",
    "legislature",
    "session",
    "nomFichierJo",
    "presidentSeance",
    "point_titre",
    "point_type",
    "valeur_ptsodj",
    "ordinal_prise",
    "ordre_absolu_seance",
    "id_acteur",
    "id_mandat",
    "code_grammaire",
    "code_style",
    "code_parole",
    "id_syceron",
    "roledebat",
    "nom_orateur",
    "qualite_orateur",
    "id_orateur",
    "stime",
]

# ---------------------------------------------------------------------------
# Fonction principale
# ---------------------------------------------------------------------------


def regrouper_que_CG(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prend un DataFrame trié par (uid, ordre_absolu_seance) et retourne
    un DataFrame entrelacé :
      - lignes d'intervention fusionnées (nb_fragments >= 1)
      - lignes d'interruption conservées telles quelles (nb_fragments = NaN)

    SANS contrainte sur code_parole (contrainte plus souple que regrouper())
    """
    cols_utiles = list(dict.fromkeys(COLS_META + ["texte"]))
    work = df[cols_utiles].copy()

    work["uid_norm"] = work["uid"].fillna("").astype(str)
    work["id_acteur_norm"] = work["id_acteur"].fillna("").astype(str)
    work["code_grammaire_norm"] = work["code_grammaire"].fillna("").astype(str)
    work["code_parole_norm"] = work["code_parole"].fillna("").astype(str)  # TODO : test
    work["texte_norm"] = work["texte"].fillna("").astype(str)

    work = work.sort_values(["uid_norm", "ordre_absolu_seance"]).reset_index(drop=True)

    resultats = []  # liste finale (interventions + interruptions)
    groupe = None  # groupe en cours d'accumulation
    buffer_interruptions = []  # interruptions entre deux fragments du même orateur

    def ligne_sortie_depuis_base(base_row: dict) -> dict:
        r = {col: base_row[col] for col in cols_utiles}
        r["nb_fragments"] = pd.NA
        r["nb_interruptions_recues"] = pd.NA
        r["a_ete_interrompu"] = pd.NA
        # r["codes_gram_fragments"] = pd.NA # ie pour traçabilité si enlève condition
        # r["codes_parole_fragments"] = pd.NA # ie pour traçabilité si enlève condition
        r["id_syceron_fragments"] = pd.NA
        # r["changement_code_grammaire"] = pd.NA # ie pour traçabilité si enlève condition
        # r["changement_code_parole"] = pd.NA # ie pour traçabilité si enlève condition
        return r

    def clore_groupe(g: dict) -> dict:
        """
        Finalise un groupe. Les interruptions du buffer seront émises APRÈS dans le flux.
        """
        row = g["premiere_ligne"].copy()
        row["texte"] = " ".join(
            g["textes"]
        )  # on prend les textes norm pour éviter les NaN
        row["nb_fragments"] = g["nb_fragments"]
        row["nb_interruptions_recues"] = g["nb_interruptions_recues"]
        row["a_ete_interrompu"] = g["nb_interruptions_recues"] > 0
        # row["codes_gram_fragments"] = "|".join(g["codes_grammaire"]) # ie pour traçabilité si enlève condition
        # row["codes_parole_fragments"] = "|".join(g["codes_parole"]) # ie pour traçabilité si enlève condition
        row["id_syceron_fragments"] = "|".join(g["codes_syceron"])
        # row["changement_code_grammaire"] = len(set(g["codes_grammaire"])) > 1 # ie pour traçabilité si enlève condition
        # row["changement_code_parole"] = len(set(g["codes_parole"])) > 1 # ie pour traçabilité si enlève condition
        return row

    records = work.to_dict("records")

    for row in records:
        cg = row["code_grammaire_norm"]
        cp = row["code_parole_norm"]  # TODO : test
        acteur_str = row["id_acteur_norm"]
        uid_str = row["uid_norm"]
        syc = str(row["id_syceron"]) if pd.notna(row["id_syceron"]) else ""

        # --- Cas 1 : interruption ---
        if cg in CODES_INTERRUPTION:
            if groupe is not None:
                # L'interruption est dans le contexte d'un groupe ouvert :
                # on l'ajoute au buffer (elle sera émise si le même orateur reprend)
                buffer_interruptions.append(row)
                groupe["nb_interruptions_recues"] += 1
            else:
                # Interruption hors contexte (cas rare) : on l'émet directement
                resultats.append(ligne_sortie_depuis_base(row))
            continue

        # --- Cas 2 : intervention principale ---
        if (
            groupe is not None
            and buffer_interruptions  # on regroupe que si bien interrompu (et pas parle 2 fois de suite)
            and acteur_str != ""  # cf les nan convertis en ""
            and groupe["id_acteur"] == acteur_str
            and groupe["uid"] == uid_str
            and groupe["codes_grammaire"][-1] == cg
        ):
            # Même orateur, même séance, mêmes codes, avec interruption -> on fusionne
            # NOTE: pas de contrainte sur code_parole (contrairement à regrouper())
            groupe["textes"].append(row["texte_norm"])
            groupe["codes_grammaire"].append(cg)
            groupe["codes_parole"].append(cp)  # TODO : test
            groupe["codes_syceron"].append(syc)
            groupe["nb_fragments"] += 1
        else:
            # Nouvel orateur ou nouvelle séance ou changement de code_grammaire
            if groupe is not None:
                # Clore le groupe précédent
                resultats.append(clore_groupe(groupe))
                # Et les interruptions en buffer suivent le groupe
                for irr in buffer_interruptions:
                    resultats.append(ligne_sortie_depuis_base(irr))
                buffer_interruptions = []

            groupe = {
                "uid": uid_str,
                "id_acteur": acteur_str,
                "premiere_ligne": {col: row[col] for col in cols_utiles},
                "textes": [row["texte_norm"]],
                "codes_grammaire": [cg],
                "codes_parole": [cp],
                "codes_syceron": [syc],
                "nb_fragments": 1,
                "nb_interruptions_recues": 0,
            }
    # Clore le dernier groupe
    if groupe is not None:
        resultats.append(clore_groupe(groupe))
        for irr in buffer_interruptions:
            resultats.append(ligne_sortie_depuis_base(irr))

    return pd.DataFrame(resultats)


In [5]:
df_group_bis_CG = regrouper_que_CG(
    df
)  # ici j'avais fait en désactivant code parole pour tester si ça changeait beaucoup : pas tant que ça, mais un peu (cf ci-dessous)
print("Shape du df regroupé que CG : ", df_group_bis_CG.shape)

df_group_bis_CG.to_csv("../data/interim/TEST_INTERRUPTIONS_BIS_CG.csv", index=False)

Shape du df regroupé que CG :  (980319, 33)


## Comparaison test

In [6]:
# Charger les df si pas en mémoire
cg_cp = pd.read_csv(
    "../data/interim/TEST_INTERRUPTIONS_BIS_CG_CP.csv", low_memory=False
)
cg = pd.read_csv("../data/interim/TEST_INTERRUPTIONS_BIS_CG.csv", low_memory=False)

# Construire les pour comparer proprement (y compris NA)
cg_cmp = cg.copy()
cg_cp_cmp = cg_cp.copy()

cg_cmp["id_syceron_key"] = cg_cmp["id_syceron"].astype("string").fillna("<NA>")
cg_cp_cmp["id_syceron_key"] = cg_cp_cmp["id_syceron"].astype("string").fillna("<NA>")
cg_cmp["a_ete_interrompu_key"] = (
    cg_cmp["a_ete_interrompu"].astype("string").fillna("<NA>")
)
cg_cp_cmp["a_ete_interrompu_key"] = (
    cg_cp_cmp["a_ete_interrompu"].astype("string").fillna("<NA>")
)

# Comparaison sur id_syceron et a_ete_interrompu)
key_cols = ["id_syceron_key", "a_ete_interrompu_key"]

# Comparer les combinaisons avec effectifs
counts_cg = cg_cmp[key_cols].value_counts(dropna=False).rename("n_cg").reset_index()
counts_cg_cp = (
    cg_cp_cmp[key_cols].value_counts(dropna=False).rename("n_cg_cp").reset_index()
)

diff_combos = (
    counts_cg.merge(counts_cg_cp, on=key_cols, how="outer")
    .fillna(0)
    .astype({"n_cg": int, "n_cg_cp": int})
)

diff_combos = (
    diff_combos[diff_combos["n_cg"] != diff_combos["n_cg_cp"]]
    .sort_values(key_cols)
    .reset_index(drop=True)
)

# Cas qui existent en CG_CP mais pas en CG
only_cg_cp = diff_combos[
    (diff_combos["n_cg"] == 0) & (diff_combos["n_cg_cp"] > 0)
].copy()
ids_only_cg_cp = set(only_cg_cp["id_syceron_key"].tolist()) - {"<NA>"}

# Retrouver dans CG avec quoi ces id_syceron ont été regroupés
if "id_syceron_fragments" in cg_cmp.columns:
    cg_cmp["id_syceron_fragments_str"] = (
        cg_cmp["id_syceron_fragments"].astype("string").fillna("")
    )
else:
    cg_cmp["id_syceron_fragments_str"] = ""

# Liste des id_syceron contenus dans chaque ligne CG (via id_syceron_fragments)
cg_cmp["ids_in_group"] = cg_cmp["id_syceron_fragments_str"].apply(
    lambda s: [x for x in str(s).split("|") if x and x != "<NA>"]
)

# Combien d'ids du set ciblé sont contenus dans la ligne CG
cg_cmp["nb_ids_cibles_dans_ligne"] = cg_cmp["ids_in_group"].apply(
    lambda ids: sum(x in ids_only_cg_cp for x in ids)
)

# Lignes CG qui absorbent au moins un id présent seulement dans CG_CP
cg_grouping_suspect = cg_cmp[cg_cmp["nb_ids_cibles_dans_ligne"] > 0].copy()

# aide repérage pour check manuel
cg_grouping_suspect["nb_ids_dans_ligne"] = cg_grouping_suspect["ids_in_group"].apply(
    len
)
cg_grouping_suspect["ids_cibles_dans_ligne"] = cg_grouping_suspect[
    "ids_in_group"
].apply(lambda ids: "|".join([x for x in ids if x in ids_only_cg_cp]))

# Pour CG_CP : lignes sources de ces ids
cg_cp_sources = cg_cp_cmp[cg_cp_cmp["id_syceron_key"].isin(ids_only_cg_cp)].copy()

# Résumés
print("Nombre total de combinaisons différentes :", len(diff_combos))
print("Combinaisons présentes seulement dans CG_CP :", len(only_cg_cp))
print("Nombre d'id_syceron concernés :", len(ids_only_cg_cp))
print("Lignes CG où ces ids sont regroupés :", len(cg_grouping_suspect))
print("Lignes sources côté CG_CP :", len(cg_cp_sources))

# Exports
# diff_combos.to_csv("../data/interim/DIFF_CG_vs_CG_CP_combos.csv", index=False)
# only_cg_cp.to_csv("../data/interim/DIFF_only_in_CG_CP_combos.csv", index=False)
cg_grouping_suspect.to_csv(
    "../data/interim/LUI_CG_regroupements_suspects_depuis_CG_CP.csv", index=False
)
cg_cp_sources.to_csv("../data/interim/CG_CP_sources_ids_concernes.csv", index=False)

Nombre total de combinaisons différentes : 1552
Combinaisons présentes seulement dans CG_CP : 1552
Nombre d'id_syceron concernés : 1552
Lignes CG où ces ids sont regroupés : 1456
Lignes sources côté CG_CP : 1552


BILAN :
Vérifier, mais en gros souvent des AVIS_GVT_1_20 qui deviennent code parole_1_2, etc. Sans doute souvent une interruption puis reprise de parole avec un code mal ré-identifié, mais sans doute plus prudent de conserver CG_CP pour être plus strict et éviter des fusions foireuses. Et au pire c'est pas 1500 lignes qui vont changer le monde (1552 D'ÉCART, 981871 VS 980319). 0,15%, cassons pas trop les pieds quoi.

# À virer

In [7]:
# # TODO fonction TER
# ça c'était une autre tentative.
# Je pense que l'autre marche bien.
# Et que donc on pourra virer.
# Je garde le temps que matthias avise du reste.

# """
# ==========================
# Regroupe les lignes d'un CSV parlementaire pour fusionner les interventions
# d'un même orateur interrompues par des INTERRUPTION_*.

# Sortie : un CSV entrelacé avec :
#   - une ligne par groupe d'intervention fusionnée (texte concaténé)
#   - les lignes INTERRUPTION conservées telles quelles, placées APRÈS leur groupe
# """

# PATTERN_INTERRUPTION = "INTERRUPTION"


# # ---------------------------------------------------------------------------
# # Fonction principale (vectorisée)
# # ---------------------------------------------------------------------------


# def regrouper(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Regroupe les interventions interrompues de manière vectorisée.

#     Stratégie :
#       1. Séparer interventions principales et interruptions
#       2. Sur les interventions, détecter les ruptures de groupe
#          (changement d'acteur ou de séance) → groupe_id cumulatif
#       3. Agréger par groupe_id (texte concaténé, first pour le reste)
#       4. Propager le groupe_id aux interruptions via ffill
#       5. Reconstruire le flux : intervention puis ses interruptions,
#          triés par (uid, ordre_absolu_seance du 1er fragment, type)
#     """
#     df = df.sort_values(["uid", "ordre_absolu_seance"]).reset_index(drop=True)

#     is_interrupt = df["code_grammaire"].str.contains(PATTERN_INTERRUPTION, na=False)
#     main = df[~is_interrupt].copy()
#     interrupts = df[is_interrupt].copy()

#     # ------------------------------------------------------------------
#     # 1. Calculer les groupe_id sur les interventions principales
#     # ------------------------------------------------------------------
#     acteur_norm = main["id_acteur"].fillna("").astype(str)
#     uid_norm = main["uid"].fillna("").astype(str)
#     code_norm = main["code_grammaire"].fillna("").astype(str)

#     # Rupture si : changement d'acteur, de séance, de code_grammaire,
#     # ou acteur vide (lignes sans id_acteur ne sont jamais fusionnées)
#     rupture = (
#         (acteur_norm != acteur_norm.shift(1))
#         | (uid_norm != uid_norm.shift(1))
#         | (code_norm != code_norm.shift(1))
#         | (acteur_norm == "")
#     )
#     main["groupe_id"] = rupture.cumsum()

#     # ------------------------------------------------------------------
#     # 2. Agréger les interventions par groupe
#     # ------------------------------------------------------------------
#     agg = {
#         c: "first"
#         for c in main.columns
#         if c not in ["texte", "code_grammaire", "groupe_id"]
#     }
#     agg["texte"] = lambda s: " ".join(s.dropna().astype(str))
#     agg["code_grammaire"] = lambda s: "|".join(s.dropna().astype(str))

#     grouped = main.groupby("groupe_id", sort=False).agg(agg).reset_index(drop=True)

#     # nb_fragments
#     grouped["nb_fragments"] = main.groupby("groupe_id", sort=False).size().values

#     # codes_fragments / changement_code_grammaire / code_grammaire (1er fragment)
#     grouped.rename(columns={"code_grammaire": "codes_fragments"}, inplace=True)
#     grouped["code_grammaire"] = grouped["codes_fragments"].str.split("|").str[0]
#     grouped["changement_code_grammaire"] = grouped["codes_fragments"].apply(
#         lambda s: len(set(s.split("|"))) > 1
#     )

#     # ------------------------------------------------------------------
#     # 3. Propager groupe_id aux interruptions via ffill
#     # ------------------------------------------------------------------
#     df["groupe_id"] = pd.NA
#     df.loc[~is_interrupt, "groupe_id"] = main["groupe_id"].values
#     df["groupe_id"] = df["groupe_id"].ffill()

#     # Compter les interruptions par groupe
#     interrupt_counts = (
#         df[is_interrupt]
#         .groupby("groupe_id", sort=False)
#         .size()
#         .rename("nb_interruptions_recues")
#         .reset_index()
#     )

#     # Rattacher le groupe_id au grouped pour le merge
#     groupe_ids = main.groupby("groupe_id", sort=False)["groupe_id"].first().values
#     grouped["groupe_id"] = groupe_ids
#     grouped = grouped.merge(interrupt_counts, on="groupe_id", how="left")
#     grouped["nb_interruptions_recues"] = (
#         grouped["nb_interruptions_recues"].fillna(0).astype(int)
#     )
#     grouped["a_ete_interrompu"] = grouped["nb_interruptions_recues"] > 0
#     grouped.drop(columns="groupe_id", inplace=True)

#     # ------------------------------------------------------------------
#     # 4. Reconstruire le flux ordonné
#     # ------------------------------------------------------------------
#     # tie-breaker : 0 = intervention, 1 = interruption
#     # → intervention toujours avant ses interruptions à même ordre
#     for col in [
#         "nb_fragments",
#         "nb_interruptions_recues",
#         "a_ete_interrompu",
#         "codes_fragments",
#         "changement_code_grammaire",
#     ]:
#         interrupts[col] = pd.NA

#     grouped["_sort_ordre"] = grouped["ordre_absolu_seance"]
#     grouped["_sort_type"] = 0

#     interrupts["_sort_ordre"] = interrupts["ordre_absolu_seance"]
#     interrupts["_sort_type"] = 1

#     result = (
#         pd.concat([grouped, interrupts], ignore_index=True)
#         .sort_values(["uid", "_sort_ordre", "_sort_type"])
#         .drop(columns=["_sort_ordre", "_sort_type"])
#         .reset_index(drop=True)
#     )

#     return result


# # ---------------------------------------------------------------------------
# # Diagnostic : changements de code_grammaire pour un même acteur
# # ---------------------------------------------------------------------------


# def diagnostic_changements_code(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     À lancer sur le df ORIGINAL (avant regroupement).
#     Retourne les cas où un même acteur enchaîne deux code_grammaire différents
#     sans interruption entre eux — utile pour investiguer les <interExtraction>.
#     """
#     main = df[~df["code_grammaire"].str.contains(PATTERN_INTERRUPTION, na=False)].copy()
#     main = main.sort_values(["uid", "ordre_absolu_seance"])

#     main["prev_acteur"] = main["id_acteur"].shift(1)
#     main["prev_code"] = main["code_grammaire"].shift(1)
#     main["prev_uid"] = main["uid"].shift(1)

#     cas = main[
#         (main["id_acteur"] == main["prev_acteur"])
#         & (main["uid"] == main["prev_uid"])
#         & (main["code_grammaire"] != main["prev_code"])
#         & main["id_acteur"].notna()
#         & (main["id_acteur"] != "")
#     ][
#         [
#             "uid",
#             "ordre_absolu_seance",
#             "id_acteur",
#             "nom_orateur",
#             "prev_code",
#             "code_grammaire",
#             "texte",
#         ]
#     ].copy()

#     cas.columns = [
#         "uid",
#         "ordre",
#         "id_acteur",
#         "nom_orateur",
#         "code_precedent",
#         "code_courant",
#         "texte",
#     ]
#     return cas
